##1) Corpus Chinese Idioms

#Idiom 1

In [1]:
#CSV-> Txt 提取成语
import pandas as pd

# 读取 CSV
df = pd.read_csv("/content/idiom.csv")

# 提取成语列
idioms = df["word"].dropna()

# 保存为文本
output_file = "idioms_only.txt"
idioms.to_csv(output_file, index=False, header=False)

print(f"完成：共提取 {len(idioms)} 条成语")



完成：共提取 30895 条成语


In [3]:
from google.colab import files
files.download("idioms_only.txt")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#Idiom 2

In [6]:
#字典型json的代码

import json
import pandas as pd

json_path = "/content/成语及俗语词典.json"
csv_path = "/content/成语及俗语词典.csv"

# 读取 JSON
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

rows = []

for idiom, info in data.items():
    row = {"word": idiom}  # 成语作为一列
    for k, v in info.items():
        if isinstance(v, list):
            row[k] = "；".join(v)   # 列表转为字符串，方便CSV/Excel
        else:
            row[k] = v
    rows.append(row)

# 转为 DataFrame
df = pd.DataFrame(rows)

# 保存为 CSV
df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print(" 转换完成！")
print("成语数量：", len(df))
print("CSV 文件路径：", csv_path)
print("字段：", list(df.columns))


 转换完成！
成语数量： 30310
CSV 文件路径： /content/成语及俗语词典.csv
字段： ['word', '拼音', '简拼', '近义词', '反义词', '用法', '解释', '出处', '例子', '谒后语', '谜语', '成语故事', '链接']


In [ ]:
from google.colab import files
files.download("/content/成语及俗语词典.csv")


In [7]:
#CSV-> Txt 提取成语
import pandas as pd

# 读取 CSV
df = pd.read_csv("/content/成语及俗语词典.csv")

# 提取成语列
idioms = df["word"].dropna()

# 保存为文本
output_file = "idioms_only.txt"
idioms.to_csv(output_file, index=False, header=False)

print(f"完成：共提取 {len(idioms)} 条成语")



完成：共提取 30310 条成语


In [8]:
from google.colab import files
files.download("idioms_only.txt")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

TTL

In [10]:
import json
import re

json_path = "/content/成语及俗语词典.json"
ttl_path = "/content/idioms_ontolex.ttl"

MAX_ENTRIES = 2000  # 你可以改大/改小。先小一点更稳。

def safe_id(s: str) -> str:
    """
    把中文/符号变成一个能用在 URI 的本地 id：
    - 保留中文、字母、数字
    - 其它字符替换为下划线
    """
    return re.sub(r"[^\w\u4e00-\u9fff]+", "_", s).strip("_") or "entry"

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

items = list(data.items())[:MAX_ENTRIES]

prefixes = """@prefix ontolex: <http://www.w3.org/ns/lemon/ontolex#> .
@prefix lexinfo: <http://www.lexinfo.net/ontology/3.0/lexinfo#> .
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .
@prefix lime: <http://www.w3.org/ns/lemon/lime#> .
@prefix dct: <http://purl.org/dc/terms/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix ex: <http://example.org/idiom/> .

"""

# Lexicon 头
ttl_lines = [prefixes]
ttl_lines.append("ex:lexicon-zh a lime:Lexicon ;\n  lime:language \"zh\" ;\n")

# 先把所有 entry 挂到 lexicon 上
entry_uris = []
for idiom, _ in items:
    eid = safe_id(idiom)
    entry_uris.append(f"ex:entry-{eid}")

# lime:entry 多值串起来
if entry_uris:
    ttl_lines.append("  lime:entry " + ", ".join(entry_uris) + " .\n\n")
else:
    ttl_lines.append("  lime:entry ex:entry-empty .\n\n")

# 逐条写 Entry/Form/Sense
for idiom, info in items:
    eid = safe_id(idiom)
    entry = f"ex:entry-{eid}"
    form = f"ex:form-{eid}"
    sense = f"ex:sense-{eid}"

    pinyin = info.get("拼音")
    abbr = info.get("简拼")
    definition = info.get("解释")
    usage = info.get("用法")
    link = info.get("链接")

    syns = info.get("近义词") or []
    ants = info.get("反义词") or []

    ttl_lines.append(f"{entry} a ontolex:MultiwordExpression ;\n"
                     f"  lexinfo:partOfSpeech lexinfo:idiom ;\n"
                     f"  ontolex:canonicalForm {form} ;\n"
                     f"  ontolex:lexicalizedSense {sense} ;\n")

    if definition:
        # 注意：这里把引号转义，避免TTL语法错误
        def_esc = definition.replace("\\", "\\\\").replace("\"", "\\\"")
        ttl_lines.append(f"  skos:definition \"{def_esc}\"@zh ;\n")

    if link:
        link_esc = str(link).replace("\\", "\\\\").replace("\"", "\\\"")
        ttl_lines.append(f"  dct:source \"{link_esc}\" ;\n")

    # 先收尾 entry（最后一行分号替换成句号）
    # 先把最后一个 " ;\n" 改成 " .\n\n"
    ttl_lines[-1] = ttl_lines[-1].rstrip(";\n") + " .\n\n"

    # Form
    ttl_lines.append(f"{form} a ontolex:Form ;\n"
                     f"  ontolex:writtenRep \"{idiom}\"@zh ;\n")
    if pinyin:
        pinyin_esc = str(pinyin).replace("\\", "\\\\").replace("\"", "\\\"")
        ttl_lines.append(f"  ontolex:phoneticRep \"{pinyin_esc}\"@zh-Latn-pinyin ;\n")
    if abbr:
        abbr_esc = str(abbr).replace("\\", "\\\\").replace("\"", "\\\"")
        ttl_lines.append(f"  dct:identifier \"{abbr_esc}\" ;\n")
    ttl_lines[-1] = ttl_lines[-1].rstrip(";\n") + " .\n\n"

    # Sense
    ttl_lines.append(f"{sense} a ontolex:LexicalSense ;\n")
    if usage:
        usage_esc = str(usage).replace("\\", "\\\\").replace("\"", "\\\"")
        ttl_lines.append(f"  skos:note \"{usage_esc}\"@zh ;\n")

    # 近义词 / 反义词 作为关系（简单用 skos:related / skos:exactMatch）
    related_uris = []
    for s in syns:
        related_uris.append(f"ex:entry-{safe_id(s)}")
    for a in ants:
        # 反义词也先用 related（如果你想更严格，我们后面可以换成自定义属性 ex:antonym）
        related_uris.append(f"ex:entry-{safe_id(a)}")

    if related_uris:
        ttl_lines.append("  skos:related " + ", ".join(sorted(set(related_uris))) + " ;\n")

    ttl_lines[-1] = ttl_lines[-1].rstrip(";\n") + " .\n\n"

with open(ttl_path, "w", encoding="utf-8") as f:
    f.writelines(ttl_lines)

print("已生成 TTL：", ttl_path)
print("导出条目数：", len(items))


已生成 TTL： /content/idioms_ontolex.ttl
导出条目数： 2000


## Idiom 3

In [13]:
#清洗 json ：

import re

src = "/content/idiom.json"
fixed = "/content/idiom_fixed.json"

with open(src, "r", encoding="utf-8") as f:
    text = f.read()

# 去掉 JS 变量声明
text = re.sub(r'^\s*(var|let|const)\s+\w+\s*=\s*', '', text)
text = re.sub(r'^\s*export\s+default\s*', '', text)

# 去掉结尾分号
text = text.strip()
if text.endswith(";"):
    text = text[:-1]

with open(fixed, "w", encoding="utf-8") as f:
    f.write(text)

print("已生成修复版：", fixed)


已生成修复版： /content/idiom_fixed.json


In [15]:
#列表型json的代码

import json
import pandas as pd

json_path = "/content/idiom_fixed.json"
csv_path = "/content/idiom.csv"

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("JSON 顶层类型：", type(data))
print("条目数量：", len(data))

rows = []

for item in data:   #  直接遍历列表
    row = {}

    for k, v in item.items():
        if isinstance(v, list):
            row[k] = "；".join(v)
        else:
            row[k] = v

    rows.append(row)

df = pd.DataFrame(rows)

df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print(" 转换完成！")
print("CSV 文件路径：", csv_path)
print("字段：", list(df.columns))


JSON 顶层类型： <class 'list'>
条目数量： 30895
 转换完成！
CSV 文件路径： /content/idiom.csv
字段： ['derivation', 'example', 'explanation', 'pinyin', 'word', 'abbreviation']


In [16]:
#CSV-> Txt 提取成语
import pandas as pd

# 读取 CSV
df = pd.read_csv("/content/idiom.csv")

# 提取成语列
idioms = df["word"].dropna()

# 保存为文本
output_file = "idioms_only.txt"
idioms.to_csv(output_file, index=False, header=False)

print(f"完成：共提取 {len(idioms)} 条成语")



完成：共提取 30895 条成语


# Ontolex

In [18]:
import json
import re

json_path = "/content/idiom_fixed.json"
ttl_path = "/content/idioms_ontolex.ttl"

def safe_id(text):
    return re.sub(r"[^\w\u4e00-\u9fff]+", "_", str(text)).strip("_")

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)   # list

prefixes = """@prefix ontolex: <http://www.w3.org/ns/lemon/ontolex#> .
@prefix lexinfo: <http://www.lexinfo.net/ontology/3.0/lexinfo#> .
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .
@prefix lime: <http://www.w3.org/ns/lemon/lime#> .
@prefix dct: <http://purl.org/dc/terms/> .
@prefix ex: <http://example.org/idiom/> .

"""

lines = [prefixes]

# Lexicon
lines.append("ex:lexicon-zh a lime:Lexicon ;\n  lime:language \"zh\" ;\n")

entries = []
for item in data:
    if "word" in item:
        entries.append(f"ex:entry-{safe_id(item['word'])}")

lines.append("  lime:entry " + ", ".join(entries) + " .\n\n")

count = 0

for item in data:
    word = item.get("word")
    if not word:
        continue

    eid = safe_id(word)
    entry = f"ex:entry-{eid}"
    form = f"ex:form-{eid}"
    sense = f"ex:sense-{eid}"

    pinyin = item.get("pinyin")
    definition = item.get("explanation")
    derivation = item.get("derivation")
    example = item.get("example")
    abbr = item.get("abbreviation")

    # Entry
    lines.append(f"""{entry} a ontolex:MultiwordExpression ;
  lexinfo:partOfSpeech lexinfo:idiom ;
  ontolex:canonicalForm {form} ;
  ontolex:lexicalizedSense {sense} ;
""")

    if definition:
        lines.append(f'  skos:definition "{definition.replace("\"","\\\"")}"@zh ;\n')

    if derivation:
        lines.append(f'  dct:source "{derivation.replace("\"","\\\"")}" ;\n')

    lines[-1] = lines[-1].rstrip(";\n") + " .\n\n"

    # Form
    lines.append(f"""{form} a ontolex:Form ;
  ontolex:writtenRep "{word}"@zh ;
""")

    if pinyin:
        lines.append(f'  ontolex:phoneticRep "{pinyin}"@zh-Latn-pinyin ;\n')

    if abbr:
        lines.append(f'  dct:identifier "{abbr}" ;\n')

    lines[-1] = lines[-1].rstrip(";\n") + " .\n\n"

    # Sense
    lines.append(f"{sense} a ontolex:LexicalSense ;\n")

    if example:
        lines.append(f'  skos:example "{example.replace("\"","\\\"")}"@zh ;\n')

    lines[-1] = lines[-1].rstrip(";\n") + " .\n\n"

    count += 1

with open(ttl_path, "w", encoding="utf-8") as f:
    f.writelines(lines)

print(" TTL 生成完成:", ttl_path)
print("成语数量:", count)


 TTL 生成完成: /content/idioms_ontolex.ttl
成语数量: 30895


In [19]:
from google.colab import files
files.download("/content/idioms_ontolex.ttl")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#合并成语 Fusion des expressions idiomatiques ： idime1+idiom2+idiom3


In [20]:
files_list = [
    "/content/idioms_only1.txt",
    "/content/idioms_only2.txt",
    "/content/idioms_only3.txt"
]

all_idioms = []
seen = set()

for fname in files_list:
    with open(fname, "r", encoding="utf-8") as f:
        for line in f:
            idiom = line.strip()
            if idiom and idiom not in seen:
                seen.add(idiom)
                all_idioms.append(idiom)

output_path = "idioms_merged_unique.txt"

with open(output_path, "w", encoding="utf-8") as f:
    for idiom in all_idioms:
        f.write(idiom + "\n")

print("合并完成")
print("原始文件数:", len(files_list))
print("去重后成语数量:", len(all_idioms))
print("输出文件:", output_path)


合并完成
原始文件数: 3
去重后成语数量: 31240
输出文件: idioms_merged_unique.txt


##合并法语的idioms

In [22]:
files_list = [
    "/content/idioms.txt",
    "/content/idioms2.txt"
]

all_idioms = []
seen = set()

for fname in files_list:
    with open(fname, "r", encoding="utf-8") as f:
        for line in f:
            idiom = line.strip()
            if idiom and idiom not in seen:
                seen.add(idiom)
                all_idioms.append(idiom)

output_path = "idioms_merged_fr.txt"

with open(output_path, "w", encoding="utf-8") as f:
    for idiom in all_idioms:
        f.write(idiom + "\n")

print("合并完成")
print("原始文件数:", len(files_list))
print("去重后成语数量:", len(all_idioms))
print("输出文件:", output_path)


合并完成
原始文件数: 2
去重后成语数量: 3091
输出文件: idioms_merged_fr.txt
